# DummyClassifier와 LOT별 Z-score 비교

각 LOT 안에서 크게 벗어난 센서값을 찾은 결과가 기존 라벨과 얼마나 맞는지 확인함.

- **DummyClassifier:** 모든 행을 0으로 예측하는 기준선.
- **Z-score:** 해당 LOT의 평균에서 표준편차 3배 이상 떨어진 센서가 하나라도 있으면 그 행을 1로 예측함.
- Train, Valid, Test로 나누지 않음. LOT 전체를 보고 하는 사후 분석이며 미래 예측 성능은 아님.
- 라벨 0은 기존 규칙에서 미검출, 라벨 1은 임시 이상 후보임. 둘 다 고장 정답으로 확정된 라벨은 아님.

CSV의 라벨은 바꾸지 않으며, 행을 늘리거나 줄여 라벨 비율을 맞추지 않음.


In [ ]:
import os
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from IPython.display import display, HTML
from sklearn.dummy import DummyClassifier

available_fonts = [font.name for font in font_manager.fontManager.ttflist]
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break

plt.rcParams["axes.unicode_minus"] = False


## 1. 설정

같은 컬럼 구조의 새 CSV는 `INPUT_CSV`만 바꾸면 됨. 컬럼 이름이 바뀌면 이 셀에서 함께 바꿈.
파생 변수, 라벨 근거와 정비 기록은 Z-score 입력에 넣지 않음.


In [ ]:
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "data")):
    ROOT = os.path.abspath(os.path.join(ROOT, ".."))
if not os.path.isdir(os.path.join(ROOT, "data")):
    ROOT = os.path.abspath(os.path.join(ROOT, ".."))

INPUT_CSV = os.path.join(ROOT, "data", "processed", "cal01_base_v1.csv")
ENC = "utf-8-sig"
TIME_COL = "MEAS_DT"
LOT_COL = "LOT_NO"
GRADE_COL = "GRADE"
LABEL_COL = "LABEL"
EXCLUDED_COL = "EXCLUDED"
ROW_ID_COL = "ROW_ID"
SENSORS = [
    "CLN-IN",
    "TC-Z1", "TC-Z2", "OP-Z1", "OP-Z2",
    "TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6",
    "OP-Z3", "OP-Z4", "OP-Z5", "OP-Z6", "CP-Z4",
    "TC-Z7", "TC-Z8", "TC-Z9", "OP-Z7", "OP-Z8",
    "TC-OUT1", "TC-OUT2",
]
FALLBACK_PLOT_SENSORS = ["CP-Z4", "TC-Z6"]
Z_THRESHOLD = 3.0
# 아래 구간 목록을 좁혀 보고 싶으면 LOT 또는 구간번호를 넣음.
REVIEW_LOT = None
REVIEW_ID = None
REVIEW_MARGIN_MIN = 3
SAVE_CSV = True
SAVE_FIGURES = True

RUN_NAME = os.path.splitext(os.path.basename(INPUT_CSV))[0]
RUN_NAME += "__" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_DIR = os.path.join(ROOT, "data", "processed", "model_comparison")
FIG_DIR = os.path.join(ROOT, "figures", "namelesspark", "11_model_comparison", RUN_NAME)


In [ ]:
data = pd.read_csv(INPUT_CSV, encoding=ENC, low_memory=False)
data[TIME_COL] = pd.to_datetime(data[TIME_COL], errors="coerce")
data[LOT_COL] = data[LOT_COL].astype("string")
data[GRADE_COL] = data[GRADE_COL].astype("string").fillna("미확인")
data[LABEL_COL] = pd.to_numeric(data[LABEL_COL], errors="coerce")
data[SENSORS] = data[SENSORS].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

excluded_text = data[EXCLUDED_COL].astype("string").str.strip().str.lower()
data["_EXCLUDED"] = ~excluded_text.isin(["false", "0", "0.0"])
data["_EXCLUSION_UNKNOWN"] = ~excluded_text.isin(["true", "false", "1", "0", "1.0", "0.0"])
data["_KEY_OK"] = data[TIME_COL].notna() & data[LOT_COL].notna()
if ROW_ID_COL and ROW_ID_COL in data.columns:
    data["_ROW_ID"] = data[ROW_ID_COL].astype("string")
else:
    data["_ROW_ID"] = data[LOT_COL] + "_" + data[TIME_COL].astype("string")
data = data.sort_values([TIME_COL, LOT_COL]).reset_index(drop=True)

display(pd.DataFrame({
    "원본 행": [len(data)], "시간 또는 LOT 누락": [int((~data["_KEY_OK"]).sum())],
    "읽을 수 없는 제외 표시": [int(data["_EXCLUSION_UNKNOWN"].sum())],
}))


## 2. LOT별 통계와 함께 비교할 행 만들기

제외 표시가 없는 행에서 LOT별 평균과 표준편차를 먼저 구함. 라벨과 시각 유무로 통계 행을 줄이지 않음.
그다음 시각과 라벨 0/1이 있고 **21개 점수가 모두 계산되는 행**을 함께 비교함.
결측이나 표준편차 0을 0점으로 채우지 않음. LOT가 없거나 제외 표시를 읽을 수 없는 행도 빼고, 아래에 빠진 행과 라벨 1 수를 표시함.


In [ ]:
reference_data = data.loc[~data["_EXCLUDED"] & data[LOT_COL].notna()].copy()
lot_mean = reference_data.groupby(LOT_COL)[SENSORS].mean()
lot_std = reference_data.groupby(LOT_COL)[SENSORS].std(ddof=0)
lot_count = reference_data.groupby(LOT_COL)[SENSORS].count()
z_scores = pd.DataFrame(index=reference_data.index)
reference_rows = []

for sensor in SENSORS:
    mean = reference_data[LOT_COL].map(lot_mean[sensor])
    std = reference_data[LOT_COL].map(lot_std[sensor]).replace(0, np.nan)
    z_scores[sensor] = reference_data[sensor].subtract(mean).divide(std).abs()

    for lot in lot_mean.index:
        std_value = lot_std.loc[lot, sensor]
        reference_rows.append({
            LOT_COL: lot, "센서": sensor,
            "통계에 사용한 행": int(lot_count.loc[lot, sensor]),
            "평균": lot_mean.loc[lot, sensor], "표준편차": std_value,
            "점수 계산 가능": bool(np.isfinite(std_value) and std_value > 0),
        })

lot_reference = pd.DataFrame(reference_rows, columns=[
    LOT_COL, "센서", "통계에 사용한 행", "평균", "표준편차", "점수 계산 가능",
])
score_ok = np.isfinite(z_scores.to_numpy()).all(axis=1)
data["_SCORE_OK"] = False
data.loc[reference_data.index, "_SCORE_OK"] = score_ok
data["_EVALUATED"] = ~data["_EXCLUDED"] & data["_KEY_OK"] & data[LABEL_COL].isin([0, 1]) & data["_SCORE_OK"]
data["_LABEL1_BEFORE"] = ~data["_EXCLUDED"] & data[LABEL_COL].eq(1)
data["_LABEL1_USED"] = data["_EVALUATED"] & data[LABEL_COL].eq(1)
data["_KNOWN_LABEL"] = ~data["_EXCLUDED"] & data[LABEL_COL].isin([0, 1])

lot_summary = data.groupby(LOT_COL, sort=False).agg(
    시작=(TIME_COL, "min"), 끝=(TIME_COL, "max"), 강종=(GRADE_COL, "first"),
    전체행=(TIME_COL, "size"), 제외표시=("_EXCLUDED", "sum"),
    제외표시없이_라벨있는행=("_KNOWN_LABEL", "sum"), 공통비교행=("_EVALUATED", "sum"),
    비교전_라벨1=("_LABEL1_BEFORE", "sum"), 비교한_라벨1=("_LABEL1_USED", "sum"),
).reset_index().sort_values(["시작", LOT_COL])
lot_summary["비교에서_빠진행"] = lot_summary["제외표시없이_라벨있는행"] - lot_summary["공통비교행"]
lot_summary["비교에서_빠진라벨1"] = lot_summary["비교전_라벨1"] - lot_summary["비교한_라벨1"]
display(lot_summary)
display(lot_reference.loc[~lot_reference["점수 계산 가능"]])


## 3. 같은 행에 두 기준선 적용하기

Z-score는 `|센서값 - 해당 LOT 평균| / 해당 LOT 표준편차`임.
21개 중 하나라도 3 이상이면 그 행에 경보를 냄. 센서를 많이 보면 경보가 늘 수 있으며, 이 점수는 이상 확률이 아님.

DummyClassifier는 항상 0을 내도록 고정함. 두 기준선 모두 라벨을 학습하거나 임계값을 고르지 않음.


In [ ]:
predictions = data[["_ROW_ID", TIME_COL, LOT_COL, GRADE_COL, LABEL_COL,
                    "_EXCLUDED", "_EXCLUSION_UNKNOWN", "_KEY_OK", "_SCORE_OK", "_EVALUATED"]].copy()
predictions["MAX_ABS_Z"] = np.nan
predictions["Z_PRED"] = np.nan
predictions["DUMMY_PRED"] = np.nan
predictions["DETECTED_SENSORS"] = ""
eligible = data.loc[data["_EVALUATED"]]

if not eligible.empty:
    valid_z = z_scores.loc[eligible.index]
    predictions.loc[eligible.index, "MAX_ABS_Z"] = valid_z.max(axis=1)
    predictions.loc[eligible.index, "Z_PRED"] = valid_z.ge(Z_THRESHOLD).any(axis=1).astype(int)
    dummy = DummyClassifier(strategy="constant", constant=0)
    # 고정 0 규칙을 설정함. 센서 관계나 평가 라벨은 학습하지 않음.
    dummy.fit([[0]], [0])
    predictions.loc[eligible.index, "DUMMY_PRED"] = dummy.predict(np.zeros((len(eligible), 1)))

    for sensor in SENSORS:
        hit_rows = valid_z.index[valid_z[sensor].ge(Z_THRESHOLD)]
        predictions.loc[hit_rows, "DETECTED_SENSORS"] += sensor + ";"
    predictions["DETECTED_SENSORS"] = predictions["DETECTED_SENSORS"].str.rstrip(";")
else:
    display(pd.DataFrame({"분석 상태": ["공통 비교 행이 없음, LOT별 원자료와 빠진 행 수만 확인"]}))


## 4. 전체 결과와 LOT 순서로 비교하기

- **Precision:** 1로 잡은 행 중 기존 라벨도 1인 비율. `찾은 1 / (찾은 1 + 기존 0인데 잡힌 행)`.
- **Recall:** 기존 라벨 1 중 찾아낸 비율. `찾은 1 / (찾은 1 + 놓친 1)`.
- 경보가 없으면 Precision, 라벨 1이 없으면 Recall과 F1은 비워 둠.
- 기존 라벨 0에 경보를 낸 비율도 함께 봄. 라벨 0은 정상 확정이 아니므로 실제 오경보 비율로 단정하지 않음.


In [ ]:
lot_order = lot_summary[LOT_COL].tolist()
comparison_rows = predictions.loc[predictions["_EVALUATED"]]
groups = [("전체", comparison_rows)]
for lot in lot_order:
    groups.append((lot, comparison_rows.loc[comparison_rows[LOT_COL].eq(lot)]))

metric_rows = []
for lot, group in groups:
    y = group[LABEL_COL].to_numpy()
    n0, n1 = int((y == 0).sum()), int((y == 1).sum())
    for model, column in [("DummyClassifier", "DUMMY_PRED"), ("Z-score", "Z_PRED")]:
        pred = group[column].to_numpy()
        tp = int(((y == 1) & (pred == 1)).sum())
        fp = int(((y == 0) & (pred == 1)).sum())
        fn = int(((y == 1) & (pred == 0)).sum())
        tn = int(((y == 0) & (pred == 0)).sum())
        note = "라벨 1이 없어 Recall, F1 생략" if n1 == 0 else ""
        if len(group) == 0:
            note = "비교 가능한 행 없음"

        metric_rows.append({
            LOT_COL: lot, "모델": model, "비교행": len(group), "라벨0": n0, "라벨1": n1,
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "Precision": tp / (tp + fp) if tp + fp else np.nan,
            "Recall": tp / n1 if n1 else np.nan,
            "F1": 2 * tp / (2 * tp + fp + fn) if n1 else np.nan,
            "라벨0 경보 비율": fp / n0 if n0 else np.nan,
            "해석할 때": note,
        })

metrics = pd.DataFrame(metric_rows)
table_columns = ["모델", "라벨1", "TP", "FN", "FP", "Precision", "Recall", "F1", "라벨0 경보 비율"]
table_labels = ["기준선", "기존 1", "찾은 1", "놓친 1", "0 중 경보", "Precision", "Recall", "F1", "0 경보 비율"]
table_widths = [0.18, 0.08, 0.08, 0.08, 0.10, 0.12, 0.12, 0.10, 0.14]
summary_table = metrics.loc[metrics[LOT_COL].eq("전체"), table_columns].copy()
for column in ["라벨1", "TP", "FN", "FP"]:
    summary_table[column] = summary_table[column].map(lambda value: f"{value:,}")
for column in ["Precision", "Recall", "F1", "라벨0 경보 비율"]:
    summary_table[column] = summary_table[column].map(lambda value: "—" if pd.isna(value) else f"{value:.1%}")

fig, axes = plt.subplots(3, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [1, 2, 2]})
axes[0].axis("off")
table = axes[0].table(cellText=summary_table.values, colLabels=table_labels,
                     colWidths=table_widths, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.7)
total_n0 = int(metrics.loc[metrics[LOT_COL].eq("전체"), "라벨0"].iloc[0])
axes[0].set_title(f"전체 비교 {len(comparison_rows):,}행, 기존 라벨 0 {total_n0:,}행")
for ax, metric, title in [
    (axes[1], "Recall", "기존 라벨 1 중 찾은 비율"),
    (axes[2], "라벨0 경보 비율", "기존 라벨 0에 경보를 낸 비율"),
]:
    table = metrics.loc[metrics[LOT_COL].ne("전체")].pivot(index=LOT_COL, columns="모델", values=metric)
    table = table.reindex(lot_order)[["DummyClassifier", "Z-score"]]
    table.plot.bar(ax=ax, color=["#7C8999", "#D79122"], rot=0)
    ax.set_title(title + " (빈칸은 라벨 또는 비교 행 부족)")
    ax.set_ylim(0, 1.08)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
    ax.grid(axis="y", alpha=0.2)
    for position, lot in enumerate(lot_order):
        if table.loc[lot].isna().all():
            ax.text(position, 0.03, "—", ha="center")

axes[-1].set_xlabel("LOT 시작 시각 순서, —는 경보 또는 해당 라벨이 없어 지표를 생략한 경우")
fig.tight_layout()
if SAVE_FIGURES:
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(os.path.join(FIG_DIR, "01_all_lots.png"), dpi=150, bbox_inches="tight")
plt.show()


## 5. LOT별 결과 바로 아래에서 같은 시각 보기

센서 그림은 **Z-score 경보가 가장 많이 나온 센서**와 **기존 라벨 1에서 가장 큰 Z-score가 나온 센서**를 골라 최대 2개 보여줌.
두 센서가 같으면 한 개만 그림. 경보가 없으면 가장 큰 점수가 나온 센서를, 비교 행이 없으면 설정한 기본 센서를 보여줌.

노란 배경은 공통 비교 행의 기존 라벨 1, 아래 점은 Z-score 경보 시각임.
센서값과 점수는 원래 기록 간격 그대로 사용함. 측정 간격이 해당 LOT의 보통 간격보다 1.5배 넘게 벌어지면 선과 구간을 끊음.
표준편차가 0이거나 결측이 있어 비교하지 못한 행은 정상 0으로 예측한 행이 아님.


In [ ]:
interval_rows = []
for lot in lot_order:
    part = data.loc[data[LOT_COL].eq(lot) & data["_KEY_OK"]].copy()
    if part.empty:
        display(metrics.loc[metrics[LOT_COL].eq(lot)])
        continue
    pred_part = predictions.loc[part.index]
    eval_index = part.index[part["_EVALUATED"]]
    gaps = part[TIME_COL].diff()
    positive_gaps = gaps.loc[gaps.gt(pd.Timedelta(0))]
    usual_gap = positive_gaps.median() if len(positive_gaps) else pd.Timedelta(seconds=1)
    gap_break = gaps.gt(usual_gap * 1.5)
    line_groups = gap_break.cumsum()
    spans = []

    for kind, mask in [
        ("기존 라벨 1", part["_EVALUATED"] & part[LABEL_COL].eq(1)),
        ("Z-score 경보", pred_part["Z_PRED"].eq(1)),
    ]:
        runs = (mask.ne(mask.shift()) | gap_break).cumsum()
        for _, run in part.loc[mask].groupby(runs.loc[mask]):
            start, end = run[TIME_COL].iloc[0], run[TIME_COL].iloc[-1]
            interval_rows.append({LOT_COL: lot, "구분": kind, "시작": start, "끝": end, "행수": len(run)})
            if kind == "기존 라벨 1":
                spans.append((start, end))

    plot_sensors = []
    if len(eval_index):
        lot_z = z_scores.loc[eval_index]
        hit_counts = lot_z.ge(Z_THRESHOLD).sum()
        first_sensor = hit_counts.idxmax() if hit_counts.max() else lot_z.max().idxmax()
        plot_sensors.append(first_sensor)
        label1_index = part.index[part["_EVALUATED"] & part[LABEL_COL].eq(1)]
        if len(label1_index):
            second_sensor = z_scores.loc[label1_index].max().idxmax()
            if second_sensor not in plot_sensors:
                plot_sensors.append(second_sensor)
    else:
        plot_sensors = [sensor for sensor in FALLBACK_PLOT_SENSORS if sensor in SENSORS]
        if not plot_sensors:
            plot_sensors = SENSORS[:2]

    lot_metrics = metrics.loc[metrics[LOT_COL].eq(lot)]
    display_table = lot_metrics[table_columns].copy()
    for column in ["라벨1", "TP", "FN", "FP"]:
        display_table[column] = display_table[column].map(lambda value: f"{value:,}")
    for column in ["Precision", "Recall", "F1", "라벨0 경보 비율"]:
        display_table[column] = display_table[column].map(lambda value: "—" if pd.isna(value) else f"{value:.1%}")

    panel_count = 1 + len(plot_sensors) + (2 if len(eval_index) else 0)
    fig, axes = plt.subplots(panel_count, 1, figsize=(14, 2.1 * panel_count))
    axes = np.atleast_1d(axes)
    axes[0].axis("off")
    table = axes[0].table(cellText=display_table.values, colLabels=table_labels,
                         colWidths=table_widths, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.7)
    axes[0].set_title("—는 경보 또는 해당 라벨이 없어 지표를 생략한 경우")
    plot_axes = axes[1:]
    for ax in plot_axes[1:]:
        ax.sharex(plot_axes[0])

    for panel, sensor in enumerate(plot_sensors):
        for _, segment in part.groupby(line_groups):
            plot_axes[panel].plot(segment[TIME_COL], segment[sensor], color="#34699A",
                                  linewidth=0.8, marker=".", markersize=1)
        unit = "\n기록값(단위 미확인)" if sensor == "CP-Z4" else ""
        plot_axes[panel].set_ylabel(sensor + unit)

    if len(eval_index):
        for _, segment in pred_part.groupby(line_groups):
            axes[-2].plot(segment[TIME_COL], segment["MAX_ABS_Z"], color="#9F6618", linewidth=0.8)
        axes[-2].axhline(Z_THRESHOLD, color="#C24536", linestyle="--", linewidth=1)
        axes[-2].set_ylabel("21개 센서 중\n가장 큰 |Z|")
        for row, column, color in [(2, LABEL_COL, "#CBA51D"), (1, "DUMMY_PRED", "#7C8999"), (0, "Z_PRED", "#D79122")]:
            times = pred_part.loc[pred_part["_EVALUATED"] & pred_part[column].eq(1), TIME_COL]
            axes[-1].scatter(times, np.repeat(row, len(times)), marker="|", s=30, color=color)
        axes[-1].set_yticks([0, 1, 2], ["Z-score 경보", "Dummy 경보", "기존 라벨 1"])
        axes[-1].set_ylim(-0.6, 2.6)

        for ax in plot_axes[:-1]:
            for start, end in spans:
                if start == end:
                    ax.axvline(start, color="#E9C94A", alpha=0.4, linewidth=1)
                else:
                    ax.axvspan(start, end, color="#E9C94A", alpha=0.25)

    for ax in plot_axes:
        ax.grid(alpha=0.2)
    for ax in plot_axes[:-1]:
        ax.tick_params(labelbottom=False)
    axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
    axes[-1].set_xlabel("실제 날짜와 시각")
    axes[-1].set_xlim(part[TIME_COL].iloc[0], part[TIME_COL].iloc[-1])
    title = "LOT " + str(lot) + " / " + str(part[GRADE_COL].iloc[0])
    title += f" / 비교 {len(eval_index):,}행"
    title += f" / 기존 라벨 0 {int(lot_metrics['라벨0'].iloc[0]):,}행"
    title += "\n" + part[TIME_COL].iloc[0].strftime("%Y-%m-%d %H:%M:%S")
    title += " ~ " + part[TIME_COL].iloc[-1].strftime("%Y-%m-%d %H:%M:%S")
    title += "\n센서선 = 전체 원자료, 노란 배경과 하단 경보 = 공통 비교 행만" if len(eval_index) else "\n비교 가능한 행 없음, 원자료만 표시"
    fig.suptitle(title)
    fig.tight_layout()
    if SAVE_FIGURES:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, "LOT_" + str(lot) + ".png"), dpi=150, bbox_inches="tight")
    plt.show()

intervals = pd.DataFrame(interval_rows, columns=[LOT_COL, "구분", "시작", "끝", "행수"])


## 6. 기존 라벨 0인데 Z-score가 잡은 구간 찾기

표의 시작과 끝은 **실제 날짜와 시각**임. 기존 라벨이 0이고 Z-score가 1인 행만 연속 구간으로 묶음.
중간에 다른 판정, 제외 행, 결측 행이 끼거나 보통 간격의 1.5배를 넘으면 끊음. 한 행짜리 구간은 시작과 끝이 같음.

아래 목록은 생략 없이 스크롤로 볼 수 있음. `REVIEW_LOT`에 LOT를 넣어 좁히고, 표의 `구간번호`를 `REVIEW_ID`에 넣으면 그 구간을 그림으로 확인함.
둘 다 비워 두면 전체 목록과 행 수가 가장 많은 구간을 보여줌. 센서 목록은 구간 안에서 한 번이라도 기준을 넘은 센서임.


In [ ]:
review_records = []
row_window_id = pd.Series("", index=predictions.index)

for lot in lot_order:
    part = predictions.loc[predictions[LOT_COL].eq(lot) & predictions["_KEY_OK"]]
    gaps = part[TIME_COL].diff()
    usual_gap = gaps.loc[gaps.gt(pd.Timedelta(0))].median()
    mask = part["_EVALUATED"] & part[LABEL_COL].eq(0) & part["Z_PRED"].eq(1)
    runs = (mask.ne(mask.shift()) | gaps.gt(usual_gap * 1.5)).cumsum()

    for number, (_, run) in enumerate(part.loc[mask].groupby(runs.loc[mask]), start=1):
        window_id = str(lot) + "-" + str(number).zfill(4)
        start, end = run[TIME_COL].iloc[0], run[TIME_COL].iloc[-1]
        detected = run["DETECTED_SENSORS"].str.split(";").explode().unique()
        sensors = []
        for sensor in SENSORS:
            if sensor in detected:
                sensors.append(sensor)

        review_records.append({
            "구간번호": window_id, LOT_COL: lot, GRADE_COL: run[GRADE_COL].iloc[0],
            "시작": start, "끝": end, "시작부터 끝까지(초)": (end - start).total_seconds(),
            "행 수": len(run), "기준 넘은 센서": ", ".join(sensors),
            "가장 큰 |Z|": run["MAX_ABS_Z"].max(),
        })
        row_window_id.loc[run.index] = window_id

review_windows = pd.DataFrame(review_records, columns=[
    "구간번호", LOT_COL, GRADE_COL, "시작", "끝", "시작부터 끝까지(초)",
    "행 수", "기준 넘은 센서", "가장 큰 |Z|",
])
review_rows = predictions.loc[row_window_id.ne("")].copy()
review_rows.insert(0, "구간번호", row_window_id.loc[review_rows.index])
review_counts = review_windows.groupby(LOT_COL, sort=False).agg(
    구간수=("구간번호", "size"), 행수=("행 수", "sum"),
).reindex(lot_order, fill_value=0).reset_index()
display(review_counts)
print(f"전체 {len(review_windows):,}구간, {len(review_rows):,}행")

# GitHub에서도 LOT별 행 수와 구간 수를 바로 볼 수 있게 그림으로 저장함.
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis("off")
count_table = review_counts.copy()
for column in ["구간수", "행수"]:
    count_table[column] = count_table[column].map(lambda value: f"{value:,}")
table = ax.table(cellText=count_table.values, colLabels=["LOT", "구간 수", "기존 0인데 잡힌 행"],
                 cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.6)
ax.set_title(f"기존 라벨 0인데 Z-score가 잡은 {len(review_rows):,}행 / {len(review_windows):,}구간")
fig.tight_layout()
if SAVE_FIGURES:
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(os.path.join(FIG_DIR, "02_label0_z1_lots.png"), dpi=150, bbox_inches="tight")
plt.show()

shown_windows = review_windows
if REVIEW_LOT is not None:
    shown_windows = shown_windows.loc[shown_windows[LOT_COL].eq(str(REVIEW_LOT))]
display(HTML('<div style="max-height:420px;overflow:auto">' +
             shown_windows.round({"가장 큰 |Z|": 2}).to_html(index=False, escape=True) + "</div>"))


In [ ]:
if REVIEW_ID is None:
    selected = shown_windows.sort_values("행 수", ascending=False, kind="stable").head(1)
else:
    selected = shown_windows.loc[shown_windows["구간번호"].eq(REVIEW_ID)]

if selected.empty:
    print("선택한 LOT 또는 구간번호에 해당하는 구간이 없음.")
else:
    display(selected)
    window = selected.iloc[0]
    start, end = window["시작"], window["끝"]
    margin = pd.Timedelta(minutes=REVIEW_MARGIN_MIN)
    lot_data = data.loc[data[LOT_COL].eq(window[LOT_COL]) & data["_KEY_OK"]]
    lot_gaps = lot_data[TIME_COL].diff()
    usual_gap = lot_gaps.loc[lot_gaps.gt(pd.Timedelta(0))].median()
    line_groups = lot_gaps.gt(usual_gap * 1.5).cumsum()
    part = lot_data.loc[lot_data[TIME_COL].between(start - margin, end + margin)]
    sensors = window["기준 넘은 센서"].split(", ")

    fig, axes = plt.subplots(len(sensors) + 1, 1, figsize=(14, 2.1 * (len(sensors) + 1)), sharex=True)
    for ax, sensor in zip(axes[:-1], sensors):
        for _, segment in part.groupby(line_groups.loc[part.index]):
            ax.plot(segment[TIME_COL], segment[sensor], color="#34699A", linewidth=1)
        mean = lot_mean.loc[window[LOT_COL], sensor]
        std = lot_std.loc[window[LOT_COL], sensor]
        ax.axhline(mean - Z_THRESHOLD * std, color="#C24536", linestyle="--", linewidth=0.8)
        ax.axhline(mean + Z_THRESHOLD * std, color="#C24536", linestyle="--", linewidth=0.8)
        unit = "\n기록값(단위 미확인)" if sensor == "CP-Z4" else ""
        ax.set_ylabel(sensor + unit)

    for _, segment in predictions.loc[part.index].groupby(line_groups.loc[part.index]):
        axes[-1].plot(segment[TIME_COL], segment["MAX_ABS_Z"], color="#9F6618")
    axes[-1].axhline(Z_THRESHOLD, color="#C24536", linestyle="--", linewidth=0.8)
    axes[-1].set_ylabel("21개 중\n가장 큰 |Z|")
    for ax in axes:
        if start == end:
            ax.axvline(start, color="#E9C94A", linewidth=2)
        else:
            ax.axvspan(start, end, color="#E9C94A", alpha=0.3)
        ax.grid(alpha=0.2)

    axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M:%S"))
    axes[-1].set_xlim(part[TIME_COL].min(), part[TIME_COL].max())
    axes[-1].set_xlabel("실제 날짜와 시각")
    fig.suptitle("구간 " + window["구간번호"] + f" / 기존 0인데 잡힌 {window['행 수']:,}행\n" +
                 start.strftime("%Y-%m-%d %H:%M:%S") + " ~ " + end.strftime("%Y-%m-%d %H:%M:%S") +
                 "\n노란 배경 = 표에서 고른 구간, 빨간 점선 = LOT 평균에서 표준편차 3배 떨어진 값")
    fig.tight_layout()
    if SAVE_FIGURES:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, "03_label0_z1_selected.png"), dpi=150, bbox_inches="tight")
    plt.show()


## 7. 결과 저장

실행할 때마다 입력 CSV 이름과 실행 시각을 붙여 이전 결과를 보존함.
행별 결과의 빈 예측은 비교에서 제외된 행이며, 0 예측과 다름.
Z-score가 기존 라벨과 다르다고 바로 라벨을 바꾸지 않고, 해당 시각의 센서와 조업 상황을 다시 확인함.
공유한 결과는 [그림 폴더](../../figures/namelesspark/11_model_comparison/)에서 볼 수 있음.

`label0_z1_windows`는 전체 구간 목록, `label0_z1_rows`는 그 구간에 속한 개별 행임. `구간번호`로 두 표를 연결함.


In [ ]:
if SAVE_CSV:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    output_tables = {
        "metrics": metrics,
        "predictions": predictions,
        "lot_reference": lot_reference,
        "lot_counts": lot_summary,
        "intervals": intervals,
        "label0_z1_windows": review_windows,
        "label0_z1_rows": review_rows,
    }
    for name, table in output_tables.items():
        table.to_csv(os.path.join(OUTPUT_DIR, RUN_NAME + "_" + name + ".csv"), index=False, encoding=ENC)

    with open(INPUT_CSV, "rb") as file:
        input_sha256 = hashlib.sha256(file.read()).hexdigest()
    settings = {
        "input_csv": os.path.basename(INPUT_CSV),
        "input_sha256": input_sha256,
        "run_name": RUN_NAME,
        "columns": dict(time=TIME_COL, lot=LOT_COL, grade=GRADE_COL,
                        label=LABEL_COL, excluded=EXCLUDED_COL, row_id=ROW_ID_COL),
        "sensors": SENSORS,
        "z_reference": "each LOT, all nonexcluded rows before label filtering",
        "std_ddof": 0,
        "threshold": Z_THRESHOLD,
        "row_rule": "any of 21 sensors has absolute z >= threshold",
        "evaluation": "nonexcluded, label 0/1, all 21 z-scores finite",
        "dummy_strategy": "constant 0",
        "split": "none",
        "sampling": "none",
        "review_lot": REVIEW_LOT,
        "review_id": REVIEW_ID,
        "review_margin_min": REVIEW_MARGIN_MIN,
        "figure_dir": os.path.relpath(FIG_DIR, ROOT),
    }
    with open(os.path.join(OUTPUT_DIR, RUN_NAME + "_settings.json"), "w", encoding="utf-8") as file:
        json.dump(settings, file, ensure_ascii=False, indent=2)
    print("CSV와 설정:", OUTPUT_DIR, "\n파일명 시작:", RUN_NAME)
    if SAVE_FIGURES:
        print("그림:", FIG_DIR)
